# Governed research figure workflow

Set `MODE` from the lifecycle state: `planning` for `work_postprocessing`, or `final` for `figure_production`. Planning creates drafts covering all applicable data and coverage records. Final mode requires recorded researcher recommendations and writes the integrity audit.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401; registers SciencePlots styles
import seaborn as sns

In [ ]:
MODE = "planning"  # planning | final; must match current.state
TODO_ID = "TODO-XXX"
FIGURE_DIR = Path("REPLACE_WITH_FIGURE_DIRECTORY")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CLASS = "condition-merged"  # raw | condition-merged | curated
SOURCE_PATHS = [Path("REPLACE_WITH_COMPLETE_LOGICAL_SOURCE")]
CURATION_MANIFEST = None  # Required when a curated table is the source.
STABLE_KEYS = ["condition_id"]
SCIENTIFIC_COLUMNS = ["condition_id", "REPLACE_X", "REPLACE_Y"]
CONDITION_COLUMNS = ["REPLACE_CONDITION"]
STATUS_COLUMNS = ["coverage_state"]
PREDEFINED_FILTERS: dict[str, object] = {}
GRAPHICAL_METHOD_BASIS = {
    "kind": "established_convention",  # inspected_paper | established_convention
    "reference": "scatter plot with explicit condition encoding",
}
RESEARCHER_RECOMMENDATION_REF = None  # Required in final mode.
FINAL_MANUAL_CHECKS = {
    "statistical_unit_and_aggregation": False,
    "title_claim_support": False,
    "axis_scale_and_units": False,
    "legend_and_series_explanation": False,
    "missingness_and_failure_visibility": False,
    "data_schematic_separation": False,
    "rendered_legibility": False,
}  # Set each item true only after the corresponding final inspection.

In [ ]:
def read_logical_table(path: Path) -> pd.DataFrame:
    if path.is_dir() or path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Adapt the template reader for source: {path}")

def load_complete_source(paths: list[Path]) -> pd.DataFrame:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing source paths: {missing}")
    frames = [read_logical_table(path).assign(_source_path=str(path)) for path in paths]
    return pd.concat(frames, ignore_index=True)

def source_file_list(paths: list[Path]) -> list[str]:
    files: list[str] = []
    for path in paths:
        files.extend(str(p) for p in sorted(path.rglob("*")) if p.is_file()) if path.is_dir() else files.append(str(path))
    return files

def require_columns(frame: pd.DataFrame, columns: list[str], name: str) -> None:
    absent = [column for column in columns if column not in frame.columns]
    if absent:
        raise KeyError(f"{name} lacks columns: {absent}")

In [ ]:
if MODE not in {"planning", "final"}:
    raise ValueError("MODE must be planning or final")
if MODE == "final" and not RESEARCHER_RECOMMENDATION_REF:
    raise ValueError("Final mode requires the recorded researcher recommendation reference")
if SOURCE_CLASS == "curated" and not CURATION_MANIFEST:
    raise ValueError("A curated source requires its row/key-preservation manifest")

source = load_complete_source(SOURCE_PATHS)
require_columns(source, STABLE_KEYS + SCIENTIFIC_COLUMNS + CONDITION_COLUMNS, "source")
if source.duplicated(STABLE_KEYS).any():
    raise ValueError("Stable keys are not unique at the declared figure unit")
print({"source_class": SOURCE_CLASS, "rows": len(source), "stable_keys": STABLE_KEYS})

In [ ]:
coverage_columns = list(dict.fromkeys(STABLE_KEYS + CONDITION_COLUMNS + STATUS_COLUMNS))
available_coverage_columns = [column for column in coverage_columns if column in source.columns]
coverage = source[available_coverage_columns].copy()
coverage["planned_figure_id"] = "figure-01"
coverage["accounting"] = "plotted"
coverage_path = FIGURE_DIR / "figure-coverage.json"
coverage_payload = {
    "stable_keys": STABLE_KEYS,
    "scientific_columns": SCIENTIFIC_COLUMNS,
    "condition_columns": CONDITION_COLUMNS,
    "source_rows": len(source),
    "represented_rows": len(coverage),
    "records": coverage.to_dict(orient="records"),
}
coverage_path.write_text(json.dumps(coverage_payload, indent=2, default=str), encoding="utf-8")
if coverage_payload["source_rows"] != coverage_payload["represented_rows"]:
    raise ValueError("Figure coverage does not reconcile to the selected complete source")

In [ ]:
X_COLUMN = "REPLACE_X"
Y_COLUMN = "REPLACE_Y"
HUE_COLUMN = "REPLACE_CONDITION"
TITLE = "Replace with a descriptive title naming quantities and conditions"
X_LABEL = "Replace x label (unit)"
Y_LABEL = "Replace y label (unit)"
plot_data = source.copy()  # Apply only TODO-defined filters.

stage_dir = FIGURE_DIR / ("draft" if MODE == "planning" else "final")
stage_dir.mkdir(parents=True, exist_ok=True)
with plt.style.context(["science", "no-latex"]):
    fig, ax = plt.subplots(figsize=(6.4, 4.2), constrained_layout=True)
    sns.scatterplot(data=plot_data, x=X_COLUMN, y=Y_COLUMN, hue=HUE_COLUMN, ax=ax)
    ax.set(title=TITLE, xlabel=X_LABEL, ylabel=Y_LABEL)
    output_stem = stage_dir / "figure-01"
    fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(output_stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
figure_record = {
    "figure_id": "figure-01",
    "stage": MODE,
    "figure_kind": "data",
    "question_or_metric": "REPLACE_FROM_TODO",
    "source_class": SOURCE_CLASS,
    "source_paths": [str(path) for path in SOURCE_PATHS],
    "source_files": source_file_list(SOURCE_PATHS),
    "stable_keys": STABLE_KEYS,
    "columns": [X_COLUMN, Y_COLUMN, HUE_COLUMN],
    "predefined_filters": PREDEFINED_FILTERS,
    "rows_used": len(plot_data),
    "method_basis": GRAPHICAL_METHOD_BASIS,
    "researcher_recommendation_ref": RESEARCHER_RECOMMENDATION_REF,
    "title": TITLE,
    "axes": {"x": X_LABEL, "y": Y_LABEL, "scale": "linear"},
    "series_label": HUE_COLUMN,
    "outputs": [str(stage_dir / "figure-01.pdf"), str(stage_dir / "figure-01.png")],
}
manifest = {
    "todo_id": TODO_ID,
    "stage": MODE,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "figures": [figure_record],
}
manifest_name = "draft-figure-manifest.json" if MODE == "planning" else "figure_manifest.json"
(FIGURE_DIR / manifest_name).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
if MODE == "planning":
    plan = {"todo_id": TODO_ID, "figures": [figure_record], "coverage_manifest": str(coverage_path)}
    (FIGURE_DIR / "figure-plan.json").write_text(json.dumps(plan, indent=2), encoding="utf-8")

In [ ]:
if MODE == "final":
    audit_checks = {
        "source_and_condition_coverage": len(plot_data) == len(source),
        "statistical_unit_and_aggregation": FINAL_MANUAL_CHECKS["statistical_unit_and_aggregation"],
        "title_claim_support": FINAL_MANUAL_CHECKS["title_claim_support"] and TITLE.startswith("Replace") is False,
        "axis_scale_and_units": FINAL_MANUAL_CHECKS["axis_scale_and_units"] and all(label and "Replace" not in label for label in [X_LABEL, Y_LABEL]),
        "legend_and_series_explanation": FINAL_MANUAL_CHECKS["legend_and_series_explanation"] and HUE_COLUMN in plot_data.columns,
        "missingness_and_failure_visibility": FINAL_MANUAL_CHECKS["missingness_and_failure_visibility"] and all(column in source.columns for column in STATUS_COLUMNS),
        "researcher_recommendation_traceability": bool(RESEARCHER_RECOMMENDATION_REF),
        "data_schematic_separation": FINAL_MANUAL_CHECKS["data_schematic_separation"] and figure_record["figure_kind"] in {"data", "schematic"},
        "rendered_legibility": FINAL_MANUAL_CHECKS["rendered_legibility"],
    }
    audit = {
        "todo_id": TODO_ID,
        "checks": audit_checks,
        "passed": all(audit_checks.values()),
        "failed_checks": [name for name, passed in audit_checks.items() if not passed],
    }
    (FIGURE_DIR / "figure_integrity_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    if not audit["passed"]:
        raise AssertionError(f"Final figure integrity audit failed: {audit['failed_checks']}")
    audit